In [ ]:
import $ivy.`org.scalatest::scalatest:3.0.8`
import org.scalatest._

# The syntax

In [ ]:
sealed trait Expr
case class Times(op1: Expr, op2: Expr) extends Expr
case class Plus(op1: Expr, op2: Expr) extends Expr
case class Div(op1: Expr, op2: Expr) extends Expr
case class Minus(op1: Expr, op2: Expr) extends Expr
case class Sqrt(op1: Expr) extends Expr
case class Minus1(op1: Expr) extends Expr
case class Num(i: Int) extends Expr


# Standard semantics

In [ ]:
class EvalSpec(eval: Expr => Double) extends FlatSpec with Matchers{
    
    // sqrt((1+2)*3)/4
    val e1: Expr = Div(Sqrt(Times(Plus(Num(1),Num(2)), Num(3))), Num(4))
    
    "eval" should "work" in {
        eval(e1) shouldBe 0.75
    }
}

In [ ]:
import scala.math.sqrt

In [ ]:
def eval(e: Expr): Double = 
    ???

In [ ]:
run(new EvalSpec(eval))

# Non-standard semantics (I): evaluation trace

In [ ]:
class EvalTraceSpec(eval: Expr => List[Expr]) extends FlatSpec with Matchers{
    
    val e1: Expr = Div(Sqrt(Times(Plus(Num(1),Num(2)), Num(3))), Num(4))
    
    "step-by-step evaluator" should "work" in {
        eval(e1) shouldBe List(
          Div(Sqrt(Times(Plus(Num(1), Num(2)), Num(3))), Num(4)),
          Num(4),
          Sqrt(Times(Plus(Num(1), Num(2)), Num(3))),
          Times(Plus(Num(1), Num(2)), Num(3)),
          Num(3),
          Plus(Num(1), Num(2)),
          Num(2),
          Num(1)
        )
    }
}

#### Ad-hoc implementation

In [ ]:
def eval_trace_ad_hoc(e: Expr): List[Expr] = 
    ???

In [ ]:
run(new EvalTraceSpec(eval_trace_ad_hoc))

#### State monad implementation

In [ ]:
import $ivy.`org.typelevel::cats-core:2.3.0`
import cats.syntax.all._, cats.implicits._, cats.data._

https://typelevel.org/cats/datatypes/state.html

Check out these functions: 
* https://typelevel.org/cats/api/cats/data/package$$State$.html
* https://typelevel.org/cats/api/cats/syntax/FlatMapOps.html

Also the book on [Scala and cats](https://www.scalawithcats.com/dist/scala-with-cats.html): 
* [Monads](https://www.scalawithcats.com/dist/scala-with-cats.html#sec:monads)
* [State monad](https://www.scalawithcats.com/dist/scala-with-cats.html#the-state-monad)

In [ ]:
def eval_trace_state(e: Expr): List[Expr] = 
    ???

In [ ]:
run(new EvalTraceSpec(eval_trace_state))

# Non-standard semantics (I): step-by-step evaluation

In [ ]:
class StepEvalSpec(eval: Expr => List[Expr]) extends FlatSpec with Matchers{
    
    val e1: Expr = Div(Sqrt(Times(Plus(Num(1),Num(2)), Num(3))), Num(4))
    
    "step-by-step evaluator" should "work" in {
        eval(e1) shouldBe List(
          Div(Sqrt(Times(Plus(Num(1), Num(2)), Num(3))), Num(4)),
          Div(Sqrt(Times(Num(3), Num(3))), Num(4)),
          Div(Sqrt(Num(9)), Num(4)),
          Div(Num(3), Num(4)),
          Num(0))
    }
}

#### Ad-hoc implementation

In [ ]:
def step_eval(e: Expr): List[Expr] = 
    ???

In [ ]:
run(new StepEvalSpec(step_eval))

#### State monad & reader implementation

In [ ]:
def step_eval_state(e: Expr): List[Expr] = 
    ???

In [ ]:
run(new StepEvalSpec(step_eval_state))

## Parser (TBD)

In [ ]:
import $ivy.`org.scala-lang.modules::scala-parser-combinators:2.1.0`

In [ ]:
import scala.util.parsing.combinator._

In [ ]:
def parse(expression: String): Option[Expr] = ??? 

In [ ]:
class ParseSpec(parse: String => Option[Expr]) extends FlatSpec with Matchers{
    "parse" should "work" in {
        // well-formed expressions
        
        // malformed expressions
    }
}

## Serializer

In [ ]:
def unparse(e: Expr): String = 
    e match {
        case Times(e1, e2) => s"(${unparse(e1)}*${unparse(e2)})"
        case Plus(e1, e2) => s"(${unparse(e1)}+${unparse(e2)})"
        case Div(e1, e2) => s"(${unparse(e1)}:${unparse(e2)})"
        case Minus(e1, e2) => s"(${unparse(e1)}-${unparse(e2)})"
        case Sqrt(e1) => s"sqrt(${unparse(e1)})"
        case Minus1(e1) => s"-(${unparse(e1)})"
        case Num(n) => n.toString
    }